# Resource Estimation: Toffoli Gates and Logical Qubits

This notebook shows how to use `ResourceEstimator` to compute the quantum resources needed for fault-tolerant phase estimation of a periodic Hamiltonian.

Given the one-norm $\lambda$ from the previous notebook and system metadata from the HDF5 file, the estimator computes:
- **Toffoli count per block-encoding step** (the cost of a single Hamiltonian query)
- **Total logical qubit count** (system + ancilla registers)

These are the two headline numbers for any quantum resource estimate.

> **Prerequisite:** The data file `data/lcbo_2x2x2.h5` must exist. It is generated by running notebook **02_gpaw_extraction** (or provided by a collaborator).

In [1]:
import warnings
import numpy as np
from numpy.exceptions import ComplexWarning
from bloch_paw import PawReader, OneNormCalculator, ResourceEstimator

warnings.filterwarnings("ignore", category=RuntimeWarning, module="numpy")
warnings.filterwarnings("ignore", category=ComplexWarning)

DATA_FILE = "../data/lcbo_2x2x2.h5"

## Setup: Compute $\lambda$ and Rank Statistics

First, we need the one-norm $\lambda$ and average rank from `OneNormCalculator`. Then we create a `ResourceEstimator` from the same HDF5 file.

In [2]:
# Compute one-norm
reader = PawReader(DATA_FILE)
inputs = reader.to_calculator_inputs()
calc = OneNormCalculator(**inputs, thr_rank=3e-5, sv_floor=1e-12, scale_floor=1e-12)

lam = calc.lambda_one_norm()
R_avg, R0 = calc.compute_average_rank()

print(f"One-norm λ = {lam:.4f}")
print(f"Average rank R_avg = {R_avg:.2f}")
print(f"One-body rank R0 = {R0:.0f}")

One-norm λ = 110.3537
Average rank R_avg = 56.11
One-body rank R0 = 40


## Creating the ResourceEstimator

`ResourceEstimator.from_hdf5()` reads the structural parameters from the HDF5 file:
- **Nk**: number of k-points
- **Npw**: number of plane waves ($\mathbf{G} \neq 0$)
- **P**: total number of upper-triangular partial-wave pairs across all atoms
- **Nb**: number of bands per k-point
- **eta**: QROAM state-preparation parameter (conventionally 10)

From these it computes **$L = 2 (N_\text{pw} + P) N_k$**, the total number of LCU labels.

In [3]:
est = ResourceEstimator.from_hdf5(DATA_FILE)

print(f"Nk  = {est.Nk}    (k-points)")
print(f"Npw = {est.Npw}  (plane waves)")
print(f"P   = {est.P}     (partial-wave pairs)")
print(f"Nb  = {est.Nb}     (bands)")
print(f"eta = {est.eta}    (QROAM parameter)")
print(f"L   = {est.L}  (total LCU labels = 2·(Npw+P)·Nk)")

Nk  = 8    (k-points)
Npw = 511  (plane waves)
P   = 15     (partial-wave pairs)
Nb  = 5     (bands)
eta = 10    (QROAM parameter)
L   = 8416  (total LCU labels = 2·(Npw+P)·Nk)


## Parameters

The resource formulas depend on several parameters. Some are system-specific (read from HDF5 or provided); others have sensible defaults.

**System-specific (from HDF5 or `OneNormCalculator`):**

| Parameter | Symbol | Source | Meaning |
|-----------|--------|--------|---------|
| `Rl` | R^(ℓ≠0) | `compute_average_rank()` | Average two-body rank |
| `R0` | R^(0) | `compute_average_rank()` | One-body rank |
| `eta` | η | HDF5 (automatic) | QROAM state-preparation parameter (conventionally 10) |

**Precision parameters (have defaults):**

| Parameter | Symbol | Default | Meaning |
|-----------|--------|---------|---------|
| `br` | b_r | 20 | Rotation-angle register bit precision (~6 decimal digits) |
| `N1` | N_1 | 10 | One-body PREPARE phase-angle bits (~3 decimal digits) |
| `N2` | N_2 | 10 | Two-body PREPARE phase-angle bits |
| `B` | B | 20 | Givens rotation bit precision (should match br) |

The defaults are sufficient for chemical accuracy (~1.6 mHa). See the `toffoli_count_per_be` docstring for guidance on when to change them.

In `qroam='optimum'` mode (the default), the QROAM batch-size parameters (kp1, ko, kp2, kr and their uncompute variants) are computed analytically to minimize total cost.

## Toffoli Count

The Toffoli count per block-encoding step is the cost of a single query to the Hamiltonian oracle. The formula accounts for QROAM read/write costs over the LCU label table, ancilla preparation, and arithmetic operations for coefficient loading and Givens rotations. Multiply by the number of QPE iterations to get the total Toffoli cost of the algorithm.

In [4]:
# System-specific parameters
Rl  = R_avg

toffolis = est.toffoli_count_per_be(Rl=Rl, R0=R0)
print(f"Toffoli gates per block-encoding step: {toffolis:,}")

Toffoli gates per block-encoding step: 62,454


## Qubit Count

The total logical qubit count includes the system register, label registers, coefficient registers, QROAM ancillae, and the phase estimation register.

The QPE precision $\varepsilon_\text{QPE}$ determines the number of iterations $I = \lceil\pi \lambda / \varepsilon_\text{QPE}\rceil$, which in turn sizes the phase estimation register.

In [ ]:
eps_chem = 1.6e-3       # chemical accuracy in Hartree (1 kcal/mol)
eps_qpe = eps_chem / 5  # QPE share of the error budget (5 error sources)
# The 5 error sources: (1) QPE phase estimation, (2) Hamiltonian truncation,
# (3) finite basis set, (4) finite k-mesh sampling, (5) coefficient loading

qubits = est.total_qubits(Rl=Rl, R0=R0, lam=lam, eps_qpe=eps_qpe)

import math
I = math.ceil(math.pi * lam / eps_qpe)
print(f"QPE precision \u03b5_QPE = {eps_qpe:.1e}")
print(f"QPE iterations I = \u2308\u03c0\u00b7\u03bb/\u03b5_QPE\u2309 = {I:,}")
print(f"Total logical qubits: {qubits:,}")

## Parameter Sensitivity

How do the resource counts change with different parameter choices? Let's vary the rotation bit precision `br` and the QPE precision `eps_qpe`.

In [6]:
print(f"{'br':>4s}  {'Toffoli/query':>15s}  {'eps_QPE':>10s}  {'Qubits':>10s}")
print("-" * 50)

for br_val in [10, 15, 20, 25, 30]:
    t = est.toffoli_count_per_be(br=br_val, Rl=Rl, R0=R0)
    q = est.total_qubits(br=br_val, Rl=Rl, R0=R0,
                          lam=lam, eps_qpe=eps_qpe)
    print(f"{br_val:4d}  {t:15,d}  {eps_qpe:10.1e}  {q:10,d}")

  br    Toffoli/query     eps_QPE      Qubits
--------------------------------------------------
  10           62,206     3.2e-04      10,234
  15           62,332     3.2e-04      10,239
  20           62,454     3.2e-04      10,244
  25           62,573     3.2e-04      10,249
  30           62,688     3.2e-04      10,254


Now let's vary the QPE precision (with fixed br=20). Higher precision means more QPE iterations, which increases the qubit count for the phase estimation register.

In [7]:
print(f"{'eps_QPE':>10s}  {'QPE iters':>10s}  {'Toffoli/query':>15s}  {'Qubits':>10s}  {'Total Toffoli':>15s}")
print("-" * 70)

for eps in [1e-2, 1e-3, 2e-4, 1e-4]:
    t = est.toffoli_count_per_be(Rl=Rl, R0=R0)
    q = est.total_qubits(Rl=Rl, R0=R0, lam=lam, eps_qpe=eps)
    iters = math.ceil(math.pi * lam / eps)
    print(f"{eps:10.1e}  {iters:10,d}  {t:15,d}  {q:10,d}  {t * iters:15,d}")

   eps_QPE   QPE iters    Toffoli/query      Qubits    Total Toffoli
----------------------------------------------------------------------
   1.0e-02      34,669           62,454      10,234    2,165,217,726
   1.0e-03     346,687           62,454      10,240   21,651,989,898
   2.0e-04   1,733,433           62,454      10,244  108,259,824,582
   1.0e-04   3,466,865           62,454      10,246  216,519,586,710


## Optimum vs Non-Optimum QROAM

By default, `qroam='optimum'` analytically minimises the QROAM batch-size parameters. You can also provide manual values using `qroam='non-optimum'`.

Let's compare the optimum result with a manually chosen set of batch sizes.

In [8]:
# Compare optimum vs manually chosen batch sizes
t_opt = est.toffoli_count_per_be(Rl=Rl, R0=R0, qroam='optimum')

# Manual batch sizes (reasonable round numbers)
t_manual = est.toffoli_count_per_be(
    Rl=Rl, R0=R0,
    qroam='non-optimum',
    kp1=19, kp1_prime=92,
    ko=13, ko_prime=92,
    kp2=140, kp2_prime=687,
    kr=48, kr_prime=687
)

print(f"Toffoli (optimum):    {t_opt:,}")
print(f"Toffoli (manual):     {t_manual:,}")
print(f"Difference:           {t_manual - t_opt:+,} ({(t_manual/t_opt - 1)*100:+.2f}%)")
print()
print("The optimum mode uses analytically derived (non-integer) batch sizes.")
print("Manual mode requires integer batch sizes, which introduces a small overhead.")

Toffoli (optimum):    62,454
Toffoli (manual):     62,460
Difference:           +6 (+0.01%)

The optimum mode uses analytically derived (non-integer) batch sizes.
Manual mode requires integer batch sizes, which introduces a small overhead.


## Interpreting the Numbers

What do these resource counts mean in practice?

In [9]:
iters = math.ceil(math.pi * lam / eps_qpe)
total_toffoli = toffolis * iters

print("=" * 55)
print("RESOURCE ESTIMATE SUMMARY")
print("=" * 55)
print(f"System: H fcc, 2×2×2 k-mesh, 5 bands")
print(f"")
print(f"One-norm λ              = {lam:,.2f}")
print(f"QPE precision ε_QPE     = {eps_qpe:.1e}")
print(f"QPE iterations          = {iters:,}")
print(f"")
print(f"Toffoli per query       = {toffolis:,}")
print(f"Total Toffoli gates     = {total_toffoli:,}")
print(f"Logical qubits          = {qubits:,}")
print("=" * 55)
print()
print("Note: Precision parameters br, N1, N2, B use defaults")
print("(20, 10, 10, 20). See docstring for when to change.")

RESOURCE ESTIMATE SUMMARY
System: H fcc, 2×2×2 k-mesh, 5 bands

One-norm λ              = 110.35
QPE precision ε_QPE     = 3.2e-04
QPE iterations          = 1,083,396

Toffoli per query       = 62,454
Total Toffoli gates     = 67,662,413,784
Logical qubits          = 10,244

Note: Precision parameters br, N1, N2, B use defaults
(20, 10, 10, 20). See docstring for when to change.


## Summary

- `ResourceEstimator.from_hdf5()` reads structural parameters ($N_k$, $N_\text{pw}$, $P$, $N_b$, $\eta$) from the HDF5 file
- `toffoli_count_per_be()` computes the Toffoli cost of a single block-encoding query
- `total_qubits()` computes the logical qubit count for the full algorithm
- The `qroam='optimum'` mode analytically minimises QROAM batch sizes; `'non-optimum'` lets you set them manually

Next: **06_full_pipeline.ipynb** puts everything together in a complete end-to-end workflow.